# Nowify – Procrastination Model Training

This notebook Dataset EDA 

- **Data:** OULAD from `ml/dataset/` 

## Inspect local OULAD dataset (`ml/dataset/`)

The folder contains: `studentInfo.csv`, `studentAssessment.csv`, `assessments.csv`, `studentVle.csv`, `studentRegistration.csv`, `courses.csv`, `vle.csv`.  
To train on this data, run the preprocessing cells below (Steps 1–5) or run `python ml/train_model_final.py` from project root.

- EDA – Exploratory Data Analysis (all dataset files)

    The following cells load each file in `ml/dataset/` and show: **shape**, **dtypes**, **missing values**, **head**, **describe** (numeric), and **value_counts** for key categorical columns. Use this to understand OULAD structure before building features for the procrastination model.

In [ ]:
# Setup: locate ml/dataset and list files
import os
import pandas as pd

cwd = os.getcwd()
ds = None
for base in [cwd, os.path.join(cwd, ".."), os.path.join(cwd, "..", "..")]:
    candidate = os.path.join(os.path.abspath(base), "ml", "dataset")
    if os.path.isfile(os.path.join(candidate, "studentInfo.csv")):
        ds = candidate
        break
if not ds:
    raise FileNotFoundError(
        "ml/dataset not found. Set CWD to project root or ml/notebooks."
    )

print("Dataset path:", ds)
print("\nFiles and sizes:")
for f in sorted(os.listdir(ds)):
    path = os.path.join(ds, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  {f}: {size_mb:.2f} MB" if size_mb >= 0.01 else f"  {f}: <0.01 MB")

In [3]:
# --- 1. studentInfo.csv ---
# One row per student per module: demographics, education, final_result (Pass/Fail/Withdrawn)
df = pd.read_csv(os.path.join(ds, "studentInfo.csv"))
print("SHAPE:", df.shape)
print("\nDTYPES:\n", df.dtypes)
print("\nMISSING per column:\n", df.isnull().sum())
print("\nHEAD(10):")
display(df.head(10))
print("\nDESCRIBE (numeric):")
display(df.describe(include="number"))
print("\nVALUE_COUNTS: final_result")
print(df["final_result"].value_counts(dropna=False))
print("\nVALUE_COUNTS: gender")
print(df["gender"].value_counts(dropna=False))
print("\nVALUE_COUNTS: highest_education (top 5)")
print(df["highest_education"].value_counts(dropna=False).head(5))

SHAPE: (32593, 12)

DTYPES:
 code_module               str
code_presentation         str
id_student              int64
gender                    str
region                    str
highest_education         str
imd_band                  str
age_band                  str
num_of_prev_attempts    int64
studied_credits         int64
disability                str
final_result              str
dtype: object

MISSING per column:
 code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64

HEAD(10):


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass
5,AAA,2013J,38053,M,Wales,A Level or Equivalent,80-90%,35-55,0,60,N,Pass
6,AAA,2013J,45462,M,Scotland,HE Qualification,30-40%,0-35,0,60,N,Pass
7,AAA,2013J,45642,F,North Western Region,A Level or Equivalent,90-100%,0-35,0,120,N,Pass
8,AAA,2013J,52130,F,East Anglian Region,A Level or Equivalent,70-80%,0-35,0,90,N,Pass
9,AAA,2013J,53025,M,North Region,Post Graduate Qualification,NaN,55<=,0,60,N,Pass



DESCRIBE (numeric):


,id_student,num_of_prev_attempts,studied_credits
count,3.259300e+04,32593.000000,32593.000000
mean,7.066877e+05,0.163225,79.758691
std,5.491673e+05,0.479758,41.071900
min,3.733000e+03,0.000000,30.000000
25%,5.085730e+05,0.000000,60.000000
50%,5.903100e+05,0.000000,60.000000
75%,6.444530e+05,0.000000,120.000000
max,2.716795e+06,6.000000,655.000000



VALUE_COUNTS: final_result
final_result
Pass           12361
Withdrawn      10156
Fail            7052
Distinction     3024
Name: count, dtype: int64

VALUE_COUNTS: gender
gender
M    17875
F    14718
Name: count, dtype: int64

VALUE_COUNTS: highest_education (top 5)
highest_education
A Level or Equivalent          14045
Lower Than A Level             13158
HE Qualification                4730
No Formal quals                  347
Post Graduate Qualification      313
Name: count, dtype: int64


In [4]:
# --- 2. studentRegistration.csv ---
# date_registration: The date of student’s registration on the module presentation,this is the number
# of days measured relative to the start of the module-presentation
# (e.g. the negative value -30 means that the student registered to module presentation 30 days before it started).

# date_unregistration:
# The number of days (relative to the start of the module-presentation) when the student unregistered
# from a module.This field is empty for students who completed the course.
# Students who have a value here (i.e., who unregistered) have "Withdrawn" as their final_result
# in studentInfo.csv.


# Registration and unregistration dates per student per module (days relative to course start)

df = pd.read_csv(os.path.join(ds, "studentRegistration.csv"))

print("SHAPE:", df.shape)

print("\nDTYPES:\n", df.dtypes)

print("\nMISSING per column:\n", df.isnull().sum())

print("\nHEAD(10):")

display(df.head(10))

print("\nDESCRIBE (numeric):")

display(df.describe(include="number"))
print(
    "\ndate_registration: min/max (days)",
    df["date_registration"].min(),
    df["date_registration"].max(),
)
print(
    "date_unregistration non-null count(Withdrawn):",
    df["date_unregistration"].notna().sum(),
)
print(
    "\ndate_unregistration: min/max (days)",
    df["date_unregistration"].min(),
    df["date_unregistration"].max(),
)

SHAPE: (32593, 5)

DTYPES:
 code_module                str
code_presentation          str
id_student               int64
date_registration      float64
date_unregistration    float64
dtype: object

MISSING per column:
 code_module                0
code_presentation          0
id_student                 0
date_registration         45
date_unregistration    22521
dtype: int64

HEAD(10):


,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN
5,AAA,2013J,38053,-110.0,NaN
6,AAA,2013J,45462,-67.0,NaN
7,AAA,2013J,45642,-29.0,NaN
8,AAA,2013J,52130,-33.0,NaN
9,AAA,2013J,53025,-179.0,NaN



DESCRIBE (numeric):


,id_student,date_registration,date_unregistration
count,3.259300e+04,32548.000000,10072.000000
mean,7.066877e+05,-69.411300,49.757645
std,5.491673e+05,49.260522,82.460890
min,3.733000e+03,-322.000000,-365.000000
25%,5.085730e+05,-100.000000,-2.000000
50%,5.903100e+05,-57.000000,27.000000
75%,6.444530e+05,-29.000000,109.000000
max,2.716795e+06,167.000000,444.000000



date_registration: min/max (days) -322.0 167.0
date_unregistration non-null count(Withdrawn): 10072

date_unregistration: min/max (days) -365.0 444.0


In [5]:
# --- 3. assessments.csv ---
# Assessment definitions per module: type (TMA/CMA/Exam), date (deadline), weight
# Three types of assessments exist:
# Tutor Marked Assessment (TMA),Computer Marked Assessment (CMA) and Final Exam (Exam).
# date is the date of the assessment deadline.(calculated as the number of days since the start of the module-presentation.)
# weight is the weight of the assessment in the final grade.(Weight of the assessment in %. Typically, Exams are treated separately and have the weight 100%; the sum of all other assessments is 100%.)
df = pd.read_csv(os.path.join(ds, "assessments.csv"))
print("SHAPE:", df.shape)
print("\nDTYPES:\n", df.dtypes)
print("\nMISSING per column:\n", df.isnull().sum())
print("\nHEAD(15):")
display(df.head(15))
print("\nVALUE_COUNTS: assessment_type")
print(df["assessment_type"].value_counts(dropna=False))
print("\nDESCRIBE (numeric):")
display(df.describe(include="number"))

SHAPE: (206, 6)

DTYPES:
 code_module              str
code_presentation        str
id_assessment          int64
assessment_type          str
date                 float64
weight               float64
dtype: object

MISSING per column:
 code_module           0
code_presentation     0
id_assessment         0
assessment_type       0
date                 11
weight                0
dtype: int64

HEAD(15):


,code_module,code_presentation,id_assessment,assessment_type,date,weight
0,AAA,2013J,1752,TMA,19.0,10.0
1,AAA,2013J,1753,TMA,54.0,20.0
2,AAA,2013J,1754,TMA,117.0,20.0
3,AAA,2013J,1755,TMA,166.0,20.0
4,AAA,2013J,1756,TMA,215.0,30.0
5,AAA,2013J,1757,Exam,NaN,100.0
6,AAA,2014J,1758,TMA,19.0,10.0
7,AAA,2014J,1759,TMA,54.0,20.0
8,AAA,2014J,1760,TMA,117.0,20.0
9,AAA,2014J,1761,TMA,166.0,20.0



VALUE_COUNTS: assessment_type
assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64

DESCRIBE (numeric):


,id_assessment,date,weight
count,206.000000,195.000000,206.000000
mean,26473.975728,145.005128,20.873786
std,10098.625521,76.001119,30.384224
min,1752.000000,12.000000,0.000000
25%,15023.250000,71.000000,0.000000
50%,25364.500000,152.000000,12.500000
75%,34891.750000,222.000000,24.250000
max,40088.000000,261.000000,100.000000


In [6]:
# --- 4. studentAssessment.csv ---
# Submissions: id_assessment, id_student, date_submitted, is_banked,
# score (for procrastination: compare date_submitted to assessment deadline)
df = pd.read_csv(os.path.join(ds, "studentAssessment.csv"))
print("SHAPE:", df.shape)
print("\nDTYPES:\n", df.dtypes)
print("\nMISSING per column:\n", df.isnull().sum())
print("\nHEAD(10):")
display(df.head(10))
print("\nDESCRIBE (numeric):")
display(df.describe(include="number"))
print("\nScore: value_counts (binned)")
print(
    pd.cut(
        df["score"],
        bins=[0, 40, 60, 80, 100],
        labels=["0-40", "40-60", "60-80", "80-100"],
    )
    .value_counts()
    .sort_index()
)

SHAPE: (173912, 5)

DTYPES:
 id_assessment       int64
id_student          int64
date_submitted      int64
is_banked           int64
score             float64
dtype: object

MISSING per column:
 id_assessment       0
id_student          0
date_submitted      0
is_banked           0
score             173
dtype: int64

HEAD(10):


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0
5,1752,45462,20,0,70.0
6,1752,45642,18,0,72.0
7,1752,52130,19,0,72.0
8,1752,53025,9,0,71.0
9,1752,57506,18,0,68.0



DESCRIBE (numeric):


,id_assessment,id_student,date_submitted,is_banked,score
count,173912.000000,1.739120e+05,173912.000000,173912.000000,173739.000000
mean,26553.803556,7.051507e+05,116.032942,0.010977,75.799573
std,8829.784254,5.523952e+05,71.484148,0.104194,18.798107
min,1752.000000,6.516000e+03,-11.000000,0.000000,0.000000
25%,15022.000000,5.044290e+05,51.000000,0.000000,65.000000
50%,25359.000000,5.852080e+05,116.000000,0.000000,80.000000
75%,34883.000000,6.344980e+05,173.000000,0.000000,90.000000
max,37443.000000,2.698588e+06,608.000000,1.000000,100.000000



Score: value_counts (binned)
score
0-40       9215
40-60     23962
60-80     64672
80-100    75561
Name: count, dtype: int64


In [7]:
# --- 5. studentVle.csv ---
# VLE activity: id_student, id_site, date (relative days), sum_click (proxy for login/engagement)
df = pd.read_csv(os.path.join(ds, "studentVle.csv"))
print("SHAPE:", df.shape)
print("\nDTYPES:\n", df.dtypes)
print("\nMISSING per column:\n", df.isnull().sum())
print("\nHEAD(10):")
display(df.head(10))
print("\nDESCRIBE (numeric):")
display(df.describe(include="number"))
# Clicks per student (sample)
clicks_per_student = df.groupby("id_student")["sum_click"].sum()
print(
    "\nClicks per student (sample stats): count=",
    clicks_per_student.count(),
    "mean=",
    clicks_per_student.mean().round(2),
    "median=",
    clicks_per_student.median(),
)

SHAPE: (10655280, 6)

DTYPES:
 code_module            str
code_presentation      str
id_student           int64
id_site              int64
date                 int64
sum_click            int64
dtype: object

MISSING per column:
 code_module          0
code_presentation    0
id_student           0
id_site              0
date                 0
sum_click            0
dtype: int64

HEAD(10):


,code_module,code_presentation,id_student,id_site,date,sum_click
0,AAA,2013J,28400,546652,-10,4
1,AAA,2013J,28400,546652,-10,1
2,AAA,2013J,28400,546652,-10,1
3,AAA,2013J,28400,546614,-10,11
4,AAA,2013J,28400,546714,-10,1
5,AAA,2013J,28400,546652,-10,8
6,AAA,2013J,28400,546876,-10,2
7,AAA,2013J,28400,546688,-10,15
8,AAA,2013J,28400,546662,-10,17
9,AAA,2013J,28400,546890,-10,1



DESCRIBE (numeric):


,id_student,id_site,date,sum_click
count,1.065528e+07,1.065528e+07,1.065528e+07,1.065528e+07
mean,7.333336e+05,7.383234e+05,9.517400e+01,3.716946e+00
std,5.827060e+05,1.312196e+05,7.607130e+01,8.849047e+00
min,6.516000e+03,5.267210e+05,-2.500000e+01,1.000000e+00
25%,5.077430e+05,6.735190e+05,2.500000e+01,1.000000e+00
50%,5.882360e+05,7.300690e+05,8.600000e+01,2.000000e+00
75%,6.464840e+05,8.770300e+05,1.560000e+02,3.000000e+00
max,2.698588e+06,1.049562e+06,2.690000e+02,6.977000e+03



Clicks per student (sample stats): count= 26074 mean= 1518.95 median= 824.0


In [8]:
# --- 6. vle.csv ---
# VLE materials: id_site, code_module, code_presentation, activity_type (resource, url, homepage, etc.)
df = pd.read_csv(os.path.join(ds, "vle.csv"))
print("SHAPE:", df.shape)
print("\nDTYPES:\n", df.dtypes)
print("\nMISSING per column:\n", df.isnull().sum())
print("\nHEAD(15):")
display(df.head(15))
print("\nVALUE_COUNTS: activity_type")
print(df["activity_type"].value_counts(dropna=False))

SHAPE: (6364, 6)

DTYPES:
 id_site                int64
code_module              str
code_presentation        str
activity_type            str
week_from            float64
week_to              float64
dtype: object

MISSING per column:
 id_site                 0
code_module             0
code_presentation       0
activity_type           0
week_from            5243
week_to              5243
dtype: int64

HEAD(15):


,id_site,code_module,code_presentation,activity_type,week_from,week_to
0,546943,AAA,2013J,resource,NaN,NaN
1,546712,AAA,2013J,oucontent,NaN,NaN
2,546998,AAA,2013J,resource,NaN,NaN
3,546888,AAA,2013J,url,NaN,NaN
4,547035,AAA,2013J,resource,NaN,NaN
5,546614,AAA,2013J,homepage,NaN,NaN
6,546897,AAA,2013J,url,NaN,NaN
7,546678,AAA,2013J,oucontent,NaN,NaN
8,546933,AAA,2013J,resource,NaN,NaN
9,546708,AAA,2013J,oucontent,NaN,NaN



VALUE_COUNTS: activity_type
activity_type
resource          2660
subpage           1055
oucontent          996
url                886
forumng            194
quiz               127
page               102
oucollaborate       82
questionnaire       61
ouwiki              49
dataplus            28
externalquiz        26
homepage            22
glossary            21
ouelluminate        21
dualpane            20
repeatactivity       5
htmlactivity         4
sharedsubpage        3
folder               2
Name: count, dtype: int64


In [9]:
# --- 7. courses.csv ---
# Module presentations and length (days)
df = pd.read_csv(os.path.join(ds, "courses.csv"))
print("SHAPE:", df.shape)
print("\nDTYPES:\n", df.dtypes)
print("\nMISSING per column:\n", df.isnull().sum())
print("\nFULL TABLE:")
display(df)
print("\nDESCRIBE:")
display(df.describe(include="number"))
print("\nVALUE_COUNTS: code_module")
print(df["code_module"].value_counts())
print("\nVALUE_COUNTS: code_presentation")
print(df["code_presentation"].value_counts())

SHAPE: (22, 3)

DTYPES:
 code_module                     str
code_presentation               str
module_presentation_length    int64
dtype: object

MISSING per column:
 code_module                   0
code_presentation             0
module_presentation_length    0
dtype: int64

FULL TABLE:


,code_module,code_presentation,module_presentation_length
0,AAA,2013J,268
1,AAA,2014J,269
2,BBB,2013J,268
3,BBB,2014J,262
4,BBB,2013B,240
5,BBB,2014B,234
6,CCC,2014J,269
7,CCC,2014B,241
8,DDD,2013J,261
9,DDD,2014J,262



DESCRIBE:


,module_presentation_length
count,22.000000
mean,255.545455
std,13.654677
min,234.000000
25%,241.000000
50%,261.500000
75%,268.000000
max,269.000000



VALUE_COUNTS: code_module
code_module
BBB    4
DDD    4
FFF    4
EEE    3
GGG    3
AAA    2
CCC    2
Name: count, dtype: int64

VALUE_COUNTS: code_presentation
code_presentation
2014J    7
2013J    6
2014B    6
2013B    3
Name: count, dtype: int64
